# Delta Lake MERGE Assignment — Incremental Data Processing (SCD)

**Objective:** Perform incremental data processing on a customer dataset using
Delta Lake, implementing both **SCD Type 1** (overwrite-in-place) and
**SCD Type 2** (historized) merge patterns.

**Dataset:** `data/customer_master.csv` (base/master data) and
`data/customer_incremental.csv` (new + updated records arriving as a batch).

**Steps covered in this notebook**
1. Set up a Spark session with Delta Lake support
2. Load the master dataset into a Delta table
3. Clean the data (handle nulls, remove duplicates)
4. Load the incremental dataset (simulated new/updated records)
5. SCD Type 1 MERGE — update existing rows in place, insert new rows
6. SCD Type 2 MERGE — keep full history with `is_current` / `effective_date` / `end_date`
7. Validate results (row counts, duplicate checks)
8. Display final dataset and summarize findings


## Step 1: Set Up Spark Session with Delta Lake

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
print("Spark version:", spark.version)

Spark version: 4.1.0


## Step 2: Load Master Dataset into a Delta Table

In [0]:
master_path = "/Volumes/workspace/default/assignment_data/customer_master.csv"
delta_master_path = "/Volumes/workspace/default/assignment_data/delta/customer_master"

df_master_raw = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv(master_path)
)

print("Raw master row count:", df_master_raw.count())
df_master_raw.show(truncate=False)


Raw master row count: 8
+-----------+-------------+----------------------+---------+-----------+---------+
|customer_id|name         |email                 |city     |signup_date|is_active|
+-----------+-------------+----------------------+---------+-----------+---------+
|1          |Ananya Sharma|ananya.sharma@mail.com|Delhi    |2023-01-15 |1        |
|2          |Rohan Verma  |rohan.verma@mail.com  |Mumbai   |2023-02-10 |1        |
|3          |Priya Nair   |NULL                  |Bangalore|2023-03-05 |1        |
|4          |Karan Mehta  |karan.mehta@mail.com  |Pune     |2023-03-20 |0        |
|5          |Sneha Iyer   |sneha.iyer@mail.com   |NULL     |2023-04-01 |1        |
|2          |Rohan Verma  |rohan.verma@mail.com  |Mumbai   |2023-02-10 |1        |
|6          |Fatima Sheikh|fatima.sheikh@mail.com|Hyderabad|2023-04-18 |1        |
|7          |Aditya Rao   |aditya.rao@mail.com   |Chennai  |2023-05-02 |1        |
+-----------+-------------+----------------------+---------+---

## Step 3: Data Cleaning

- Drop exact duplicate rows
- Handle nulls: fill missing `email` with `"unknown"`, missing `city` with `"Unknown"`


In [0]:
dup_count = df_master_raw.count() - df_master_raw.dropDuplicates().count()
print(f"Duplicate rows found: {dup_count}")

df_master_clean = (
    df_master_raw.dropDuplicates()
    .withColumn("email", F.when(F.col("email").isNull(), F.lit("unknown")).otherwise(F.col("email")))
    .withColumn("city", F.when(F.col("city").isNull(), F.lit("Unknown")).otherwise(F.col("city")))
)

print("Cleaned master row count:", df_master_clean.count())
df_master_clean.show(truncate=False)


Duplicate rows found: 1
Cleaned master row count: 7
+-----------+-------------+----------------------+---------+-----------+---------+
|customer_id|name         |email                 |city     |signup_date|is_active|
+-----------+-------------+----------------------+---------+-----------+---------+
|4          |Karan Mehta  |karan.mehta@mail.com  |Pune     |2023-03-20 |0        |
|6          |Fatima Sheikh|fatima.sheikh@mail.com|Hyderabad|2023-04-18 |1        |
|7          |Aditya Rao   |aditya.rao@mail.com   |Chennai  |2023-05-02 |1        |
|3          |Priya Nair   |unknown               |Bangalore|2023-03-05 |1        |
|2          |Rohan Verma  |rohan.verma@mail.com  |Mumbai   |2023-02-10 |1        |
|1          |Ananya Sharma|ananya.sharma@mail.com|Delhi    |2023-01-15 |1        |
|5          |Sneha Iyer   |sneha.iyer@mail.com   |Unknown  |2023-04-01 |1        |
+-----------+-------------+----------------------+---------+-----------+---------+



In [0]:
# Write cleaned master data as a Delta table
df_master_clean.write.format("delta").mode("overwrite").save(delta_master_path)
spark.read.format("delta").load(delta_master_path).show(truncate=False)


+-----------+-------------+----------------------+---------+-----------+---------+
|customer_id|name         |email                 |city     |signup_date|is_active|
+-----------+-------------+----------------------+---------+-----------+---------+
|4          |Karan Mehta  |karan.mehta@mail.com  |Pune     |2023-03-20 |0        |
|6          |Fatima Sheikh|fatima.sheikh@mail.com|Hyderabad|2023-04-18 |1        |
|7          |Aditya Rao   |aditya.rao@mail.com   |Chennai  |2023-05-02 |1        |
|3          |Priya Nair   |unknown               |Bangalore|2023-03-05 |1        |
|2          |Rohan Verma  |rohan.verma@mail.com  |Mumbai   |2023-02-10 |1        |
|1          |Ananya Sharma|ananya.sharma@mail.com|Delhi    |2023-01-15 |1        |
|5          |Sneha Iyer   |sneha.iyer@mail.com   |Unknown  |2023-04-01 |1        |
+-----------+-------------+----------------------+---------+-----------+---------+



## Step 4: Load Incremental Dataset (new/updated records)

In [0]:
incremental_path = "/Volumes/workspace/default/assignment_data/customer_incremental.csv"

df_incremental = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv(incremental_path)
)

print("Incremental row count:", df_incremental.count())
df_incremental.show(truncate=False)


Incremental row count: 4
+-----------+------------+---------------------+---------+-----------+---------+
|customer_id|name        |email                |city     |signup_date|is_active|
+-----------+------------+---------------------+---------+-----------+---------+
|2          |Rohan Verma |rohan.verma@mail.com |Bangalore|2023-02-10 |1        |
|4          |Karan Mehta |karan.mehta@mail.com |Pune     |2023-03-20 |1        |
|8          |Neha Kapoor |neha.kapoor@mail.com |Kolkata  |2024-01-10 |1        |
|9          |Vikram Singh|vikram.singh@mail.com|Jaipur   |2024-01-12 |1        |
+-----------+------------+---------------------+---------+-----------+---------+



## Step 5: SCD Type 1 MERGE (overwrite in place)

Existing `customer_id`s get their attributes overwritten with the latest values.
New `customer_id`s are inserted. No history is kept — this mirrors a typical
"current state only" dimension table.


In [0]:
from delta.tables import DeltaTable

scd1_path = "/Volumes/workspace/default/assignment_data/delta/customer_scd1"

# Seed the SCD1 table from the cleaned master data
df_master_clean.write.format("delta").mode("overwrite").save(scd1_path)
scd1_table = DeltaTable.forPath(spark, scd1_path)

(
    scd1_table.alias("t")
    .merge(df_incremental.alias("s"), "t.customer_id = s.customer_id")
    .whenMatchedUpdate(set={
        "name": "s.name",
        "email": "s.email",
        "city": "s.city",
        "signup_date": "s.signup_date",
        "is_active": "s.is_active",
    })
    .whenNotMatchedInsert(values={
        "customer_id": "s.customer_id",
        "name": "s.name",
        "email": "s.email",
        "city": "s.city",
        "signup_date": "s.signup_date",
        "is_active": "s.is_active",
    })
    .execute()
)

print("SCD Type 1 result (current state only):")
spark.read.format("delta").load(scd1_path).orderBy("customer_id").show(truncate=False)


SCD Type 1 result (current state only):
+-----------+-------------+----------------------+---------+-----------+---------+
|customer_id|name         |email                 |city     |signup_date|is_active|
+-----------+-------------+----------------------+---------+-----------+---------+
|1          |Ananya Sharma|ananya.sharma@mail.com|Delhi    |2023-01-15 |1        |
|2          |Rohan Verma  |rohan.verma@mail.com  |Bangalore|2023-02-10 |1        |
|3          |Priya Nair   |unknown               |Bangalore|2023-03-05 |1        |
|4          |Karan Mehta  |karan.mehta@mail.com  |Pune     |2023-03-20 |1        |
|5          |Sneha Iyer   |sneha.iyer@mail.com   |Unknown  |2023-04-01 |1        |
|6          |Fatima Sheikh|fatima.sheikh@mail.com|Hyderabad|2023-04-18 |1        |
|7          |Aditya Rao   |aditya.rao@mail.com   |Chennai  |2023-05-02 |1        |
|8          |Neha Kapoor  |neha.kapoor@mail.com  |Kolkata  |2024-01-10 |1        |
|9          |Vikram Singh |vikram.singh@mail.co

## Step 6: SCD Type 2 MERGE (full history)

Adds `effective_date`, `end_date`, and `is_current` columns. When an
existing customer's attributes change, the old row is closed out
(`is_current = false`, `end_date` set) and a new row is inserted with
`is_current = true`. Brand-new customers are inserted as current rows.


In [0]:
scd2_path = "/Volumes/workspace/default/assignment_data/delta/customer_scd22"

df_scd2_seed = (
    df_master_clean
    .withColumn("effective_date", F.current_date())
    .withColumn("end_date", F.lit(None).cast("date"))
    .withColumn("is_current", F.lit(True))
)
df_scd2_seed.write.format("delta").mode("overwrite").save(scd2_path)
scd2_table = DeltaTable.forPath(spark, scd2_path)

# Identify which incremental rows actually represent a *change* vs current data
current_df = spark.read.format("delta").load(scd2_path).filter("is_current = true")
changed = (
    df_incremental.alias("s")
    .join(current_df.alias("t"), "customer_id", "left")
    .where(
        F.col("t.customer_id").isNull() |
        (F.col("s.name") != F.col("t.name")) |
        (F.col("s.email") != F.col("t.email")) |
        (F.col("s.city") != F.col("t.city")) |
        (F.col("s.is_active") != F.col("t.is_active"))
    )
    .select("s.*")
)
print("Rows representing a real change or a new customer:")
changed.show(truncate=False)

# 1) Close out old versions of changed customers
(
    scd2_table.alias("t")
    .merge(changed.alias("s"), "t.customer_id = s.customer_id AND t.is_current = true")
    .whenMatchedUpdate(set={
        "is_current": "false",
        "end_date": "current_date()",
    })
    .execute()
)

# 2) Insert new current-version rows for changed + new customers
new_versions = (
    changed
    .withColumn("effective_date", F.current_date())
    .withColumn("end_date", F.lit(None).cast("date"))
    .withColumn("is_current", F.lit(True))
)
new_versions.write.format("delta").mode("append").save(scd2_path)

print("SCD Type 2 result (full history):")
spark.read.format("delta").load(scd2_path).orderBy("customer_id", "effective_date").show(truncate=False)


Rows representing a real change or a new customer:
+-----------+------------+---------------------+---------+-----------+---------+
|customer_id|name        |email                |city     |signup_date|is_active|
+-----------+------------+---------------------+---------+-----------+---------+
|2          |Rohan Verma |rohan.verma@mail.com |Bangalore|2023-02-10 |1        |
|4          |Karan Mehta |karan.mehta@mail.com |Pune     |2023-03-20 |1        |
|8          |Neha Kapoor |neha.kapoor@mail.com |Kolkata  |2024-01-10 |1        |
|9          |Vikram Singh|vikram.singh@mail.com|Jaipur   |2024-01-12 |1        |
+-----------+------------+---------------------+---------+-----------+---------+

SCD Type 2 result (full history):
+-----------+-------------+----------------------+---------+-----------+---------+--------------+----------+----------+
|customer_id|name         |email                 |city     |signup_date|is_active|effective_date|end_date  |is_current|
+-----------+-------------

## Step 7: Validate Results

- Row counts before/after MERGE
- No duplicate `(customer_id)` among *current* rows
- SCD2 history table should have >= SCD1 row count (it keeps old versions)


In [0]:
scd1_final = spark.read.format("delta").load(scd1_path)
scd2_final = spark.read.format("delta").load(scd2_path)

print("SCD1 row count (current state):", scd1_final.count())
print("SCD2 row count (with history): ", scd2_final.count())
print("SCD2 current-only row count:   ", scd2_final.filter("is_current = true").count())

dup_customers_scd1 = (
    scd1_final.groupBy("customer_id").count().filter("count > 1").count()
)
print("Duplicate customer_ids in SCD1 table:", dup_customers_scd1)

dup_current_scd2 = (
    scd2_final.filter("is_current = true")
    .groupBy("customer_id").count().filter("count > 1").count()
)
print("Duplicate *current* customer_ids in SCD2 table:", dup_current_scd2)

assert dup_customers_scd1 == 0, "SCD1 table should have one row per customer_id"
assert dup_current_scd2 == 0, "SCD2 table should have exactly one current row per customer_id"
print("\nValidation passed: no duplicate current records in either table.")


SCD1 row count (current state): 9
SCD2 row count (with history):  11
SCD2 current-only row count:    9
Duplicate customer_ids in SCD1 table: 0
Duplicate *current* customer_ids in SCD2 table: 0

Validation passed: no duplicate current records in either table.


## Step 8: Final Output & Summary

In [0]:
print("=== Final SCD Type 1 table (current state only) ===")
scd1_final.orderBy("customer_id").show(truncate=False)

print("=== Final SCD Type 2 table (full history) ===")
scd2_final.orderBy("customer_id", "effective_date").show(truncate=False)

print("=== Delta table history (SCD1) — shows the MERGE as a versioned operation ===")
scd1_table.history().select("version", "timestamp", "operation", "operationParameters").show(truncate=False)


=== Final SCD Type 1 table (current state only) ===
+-----------+-------------+----------------------+---------+-----------+---------+
|customer_id|name         |email                 |city     |signup_date|is_active|
+-----------+-------------+----------------------+---------+-----------+---------+
|1          |Ananya Sharma|ananya.sharma@mail.com|Delhi    |2023-01-15 |1        |
|2          |Rohan Verma  |rohan.verma@mail.com  |Bangalore|2023-02-10 |1        |
|3          |Priya Nair   |unknown               |Bangalore|2023-03-05 |1        |
|4          |Karan Mehta  |karan.mehta@mail.com  |Pune     |2023-03-20 |1        |
|5          |Sneha Iyer   |sneha.iyer@mail.com   |Unknown  |2023-04-01 |1        |
|6          |Fatima Sheikh|fatima.sheikh@mail.com|Hyderabad|2023-04-18 |1        |
|7          |Aditya Rao   |aditya.rao@mail.com   |Chennai  |2023-05-02 |1        |
|8          |Neha Kapoor  |neha.kapoor@mail.com  |Kolkata  |2024-01-10 |1        |
|9          |Vikram Singh |vikram.s

### Summary of Findings

- The raw master dataset contained **1 duplicate row** and **2 null values**
  (`email`, `city`), both handled before loading into Delta.
- The incremental batch contained **2 updates** to existing customers
  (`customer_id` 2 changed city, `customer_id` 4 was reactivated) and
  **2 brand-new customers** (`customer_id` 8, 9).
- **SCD Type 1** MERGE correctly overwrote the changed attributes in place —
  the table always reflects only the latest known state per customer, with
  one row per `customer_id`.
- **SCD Type 2** MERGE preserved history: for every customer whose attributes
  changed, the old row was closed out (`is_current = false`, `end_date` set)
  and a new row was inserted (`is_current = true`), while unaffected customers
  kept their single original row.
- Delta Lake's transaction log (`DESCRIBE HISTORY`) confirms each MERGE was
  captured as an atomic, versioned operation — enabling time travel and audit
  of every change to the table.
- Validation confirmed there are **no duplicate customer_ids** among current
  records in either table, meaning both MERGE strategies preserved data
  integrity.
